In [0]:
%run ../utils/utils_feat_squad2_99_helpers

In [0]:
%pip install python-dotenv

In [0]:
import os
from dotenv import load_dotenv

load_dotenv(".env")

client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")

storage_account_name = os.getenv("STORAGE_ACCOUNT_NAME")
container_name = os.getenv("CONTAINER_NAME")

if all([
    client_id,
    tenant_id,
    client_secret,
    storage_account_name,
    container_name
]):
    print("Variáveis carregadas com sucesso!")
else:
    raise ValueError("Erro ao carregar variáveis do .env")

In [0]:
print(storage_account_name)
print(container_name)

In [0]:
base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/"
path_vendas_raw = base_path + "vendas_raw/2026/02/21/"

df_teste = (
    spark.read
    .format("binaryFile")
    .option("recursiveFileLookup", "true")
    .option(
        f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net",
        "OAuth"
    )
    .option(
        f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net",
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
    )
    .option(
        f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net",
        client_id
    )
    .option(
        f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net",
        client_secret
    )
    .option(
        f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net",
        f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
    )
    .load(path_vendas_raw)
)

display(df_teste.select("path", "length", "modificationTime"))
print("Conexão com ADLS Gen2 realizada com sucesso.")


In [0]:
inicio = log_inicio("feat_squad2_00_setup_config")

try:
    container_client = get_container_client()
    log.info("Conexão com ADLS estabelecida!")
    log.info(f"Container: {ADLS_CONTAINER}")

except Exception as e:
    log.error(f"Erro ao conectar no ADLS: {str(e)}")
    raise


try:
    itens = list(container_client.get_paths())
    log.info(f"{len(itens)} item(ns) encontrado(s) no container")

    for item in itens:
        tipo = "Pasta" if item.is_directory else "Arquivo"
        print(f"{tipo}: {item.name}")

except Exception as e:
    log.error(f"Erro ao listar container: {str(e)}")
    raise


try:
    snapshots = listar_snapshots()
    log.info(f"{len(snapshots)} snapshot(s) encontrado(s)")

    for snap in sorted(snapshots):
        print(f"Snapshot: {snap}")

    mais_recente = get_snapshot_mais_recente()
    log.info(f"Snapshot mais recente: {mais_recente}")

except Exception as e:
    log.error(f"Erro ao listar snapshots: {str(e)}")
    raise


try:
    snap_ref = get_snapshot_mais_recente()
    log.info(f"Validando tabelas no snapshot: {snap_ref}")

    for tabela in TABELAS_SQUAD2:
        try:
            df = ler_parquet(snap_ref, tabela)
            log.info(
                f"OK {tabela} → "
                f"{df.count()} linhas | "
                f"{len(df.columns)} colunas"
            )

        except Exception as e:
            log.error(f"Erro em {tabela}: {str(e)}")

except Exception as e:
    log.error(f"Erro na validação: {str(e)}")
    raise


log_fim("feat_squad2_00_setup_config", inicio)